## Calibrating and Cataloguing Green Band Images

In [25]:
# Import various libraries 
import numpy as np
import astropy
import photutils
import ccdproc
from ccdproc import CCDData, combiner
from astropy import units as u
import astropy.io.fits as fits
from astropy.io import ascii
import time
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from photutils.centroids import centroid_com, centroid_1dg, centroid_2dg
from photutils.aperture import CircularAperture
from photutils.aperture import aperture_photometry
from photutils.background import Background2D
from photutils.background import MedianBackground
from photutils.detection import DAOStarFinder
from photutils.segmentation  import detect_sources, deblend_sources, SourceCatalog
from scipy.ndimage import shift
import gc                               

from astropy.coordinates import SkyCoord
from astroquery.gaia import Gaia # this library is used to query astronomical databases 
from photutils.segmentation import SourceCatalog, SegmentationImage

In [26]:
# ignoring warnings
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("astropy").setLevel(logging.WARNING)

In [27]:
def band_sel(fidx):
    # Band Variables
    bands = ["b", "g", "r"]
    band = bands[fidx]
    fn_name = '_'+band + '.fit'
    return fn_name, band

In [28]:
fidx = 1 # selecting green band
fn_name, band = band_sel(fidx)

check if median sky subtraction is necessary before cateloguing 

In [29]:
fn = band+"_NGC_2547_cropped.fits"
scim=CCDData.read(fn,  unit="adu")   # Read the image

w = scim.wcs                                      # Read the World Coordinate System
print(w)

med=np.median(scim.data)                          # Median pixel value
scim.data=scim.data-med                           # Use the median for the sky subtraction
mean, median, std = astropy.stats.sigma_clipped_stats(scim.data, sigma=3.0, maxiters=5)    
print('Image stats (mean, median and standard deviation):', mean,median,std)

segimage=detect_sources(scim.data, 2.00*std, 9, connectivity=4, mask=None)
outfn="segimage/segimage" +fn_name                # Set the output file name
hdu=fits.PrimaryHDU(segimage)          # Define a FITS header for this data
# Convert WCS to header format
wcsheader = w.to_header(relax=True)
hdu.header.extend(wcsheader)
hdu.writeto(outfn, overwrite=True)   # Write the output and overwrite existing files if needed. 

WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN' 'DEC--TAN'
CUNIT : 'deg' 'deg'
CRVAL : 122.353311568 -48.8902271055
CRPIX : 587.693748474 -116.964008331
PC1_1 PC1_2  : -0.000595693031779 0.000280247364189
PC2_1 PC2_2  : -0.000281119408085 -0.000596031924072
CDELT : 1.0 1.0
NAXIS : 1080  700
Image stats (mean, median and standard deviation): -0.3006382 -0.74451816 9.809436


In [24]:
# ── Load image and build source catalogue ─────────────────────────────────────
with fits.open(fn) as hdul:
    sci_data   = hdul[0].data.astype(float).copy()
    sci_header = hdul[0].header

with fits.open(outfn) as hdul:
    seg_data = hdul[0].data.copy()

w = WCS(sci_header)
sci_data -= np.median(sci_data) # background subtraction 
_, _, std = astropy.stats.sigma_clipped_stats(sci_data, sigma=3.0, maxiters=5)

segmap       = SegmentationImage(seg_data)
source_table = SourceCatalog(sci_data, segmap, wcs=w)

# ── Aperture photometry on all detected sources ───────────────────────────────
positions = []
for obj in source_table:
    positions.append((obj.centroid[0], obj.centroid[1]))

apertures  = CircularAperture(positions, r=8.3)
phot_table = aperture_photometry(sci_data, apertures)

# print("\nFirst 10 rows of photometry table:")
# print(phot_table[:10])
# print("\nColumn names:", phot_table.colnames)

# ── Gaia query ────────────────────────────────────────────────────────────────
ra_field  = w.wcs.crval[0]
dec_field = w.wcs.crval[1]

coord  = SkyCoord(ra=ra_field, dec=dec_field, unit='deg', frame='icrs')
radius = u.Quantity(0.25, u.deg)

Gaia.ROW_LIMIT = 5000
print("\nQuerying Gaia DR3...")
j = Gaia.cone_search(coordinate=coord, radius=radius)

r = j.get_results()

# restricting stars that have less than 30
mask = (r['phot_g_mean_mag']  > 0) & (r['phot_g_mean_mag']  < 30) & \
       (r['phot_bp_mean_mag'] > 0) & (r['phot_bp_mean_mag'] < 30) & \
       (r['phot_rp_mean_mag'] > 0) & (r['phot_rp_mean_mag'] < 30)
r = r[mask]
r.sort('phot_g_mean_mag')

nrows, ncols = sci_data.shape
gaia_coords  = SkyCoord(ra=r['ra'], dec=r['dec'], unit='deg', frame='icrs')
px, py       = w.world_to_pixel(gaia_coords)

in_image = (px > 20) & (px < ncols-20) & (py > 20) & (py < nrows-20)
r  = r[in_image]
px = px[in_image]
py = py[in_image]
print(f"Gaia sources inside image footprint: {len(r)}")

# ── Match each Gaia star to nearest source in phot_table ─────────────────────
phot_x = np.array(phot_table['xcenter'])
phot_y = np.array(phot_table['ycenter'])

print(" Reference stars (G < 12)")
print(f"{'#':<4} {'RA':>12} {'Dec':>12} {'G_BP':>6} {'G':>6} {'G_RP':>6}  "
      f"{'x_pix':>7} {'y_pix':>7}  {'dist':>5} {'aperture_sum':>14}")
print("-" * 90)
count = 0
for i in range(len(r)):
    if r['phot_g_mean_mag'][i] < 12:
        dists      = np.sqrt((phot_x - px[i])**2 + (phot_y - py[i])**2)
        nearest    = int(np.argmin(dists))
        flux       = phot_table['aperture_sum'][nearest]
        print(f"{count:<4} {r['ra'][i]:>12.5f} {r['dec'][i]:>12.5f} "
              f"{r['phot_bp_mean_mag'][i]:>6.3f} {r['phot_g_mean_mag'][i]:>6.3f} "
              f"{r['phot_rp_mean_mag'][i]:>6.3f}  {px[i]:>7.1f} {py[i]:>7.1f}  "
              f"{dists[nearest]:>5.1f} {flux:>14.1f}")
        count += 1
        if count >= 60:   # added
            break

print("Faint stars (15 < G < 18)")  # was 14 < G < 17
print(f"{'#':<4} {'RA':>12} {'Dec':>12} {'G_BP':>6} {'G':>6} {'G_RP':>6}  "
      f"{'x_pix':>7} {'y_pix':>7}  {'dist':>5} {'aperture_sum':>14}")
print("-" * 90)
count = 0
for i in range(len(r)):
    g = r['phot_g_mean_mag'][i]
    if 15 < g < 18:                     # was 14 < g < 17
        dists   = np.sqrt((phot_x - px[i])**2 + (phot_y - py[i])**2)
        nearest = int(np.argmin(dists))
        flux    = phot_table['aperture_sum'][nearest]
        # print(f"{count:<4} {r['ra'][i]:>12.5f} {r['dec'][i]:>12.5f} "
        #       f"{r['phot_bp_mean_mag'][i]:>6.3f} {g:>6.3f} "
        #       f"{r['phot_rp_mean_mag'][i]:>6.3f}  {px[i]:>7.1f} {py[i]:>7.1f}  "
        #       f"{dists[nearest]:>5.1f} {flux:>14.1f}")
        count += 1
        if count >= 60:
            break


Querying Gaia DR3...
Gaia sources inside image footprint: 590
 Reference stars (G < 12)
#              RA          Dec   G_BP      G   G_RP    x_pix   y_pix   dist   aperture_sum
------------------------------------------------------------------------------------------
0       122.46628    -49.00695 10.776 10.703 10.562    560.4    90.4    0.2        92857.5
1       122.29084    -49.01781 12.269 11.645 10.886    725.4    30.7    0.1        29207.5
2       122.49974    -48.98119 12.338 11.752 11.023    513.5    69.3    6.0        46792.7
3       122.50595    -48.97365 11.968 11.853 11.637    503.0    61.6    7.0        46792.7
Faint stars (15 < G < 18)
#              RA          Dec   G_BP      G   G_RP    x_pix   y_pix   dist   aperture_sum
------------------------------------------------------------------------------------------
